In [28]:
import os
import boto3
from botocore.exceptions import ClientError

# --- CONFIGURATION DE L'ACCÈS S3 (MINIO) ---
os.environ['MLFLOW_TRACKING_URI'] = 'http://localhost:5000'
os.environ['MLFLOW_S3_ENDPOINT_URL'] = 'http://localhost:9000'
os.environ['AWS_ACCESS_KEY_ID'] = 'minioadmin'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'minioadmin'
os.environ['MLFLOW_S3_IGNORE_TLS'] = 'true'
# Ajout important pour éviter que boto3 ne cherche une région AWS par défaut
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1' 

# --- VÉRIFICATION & RÉPARATION AUTOMATIQUE DE MINIO ---
def check_and_create_bucket(bucket_name="mlflow"):
    print(f"🔍 Vérification de l'accès MinIO sur {os.environ['MLFLOW_S3_ENDPOINT_URL']}...")
    
    s3 = boto3.client(
        's3',
        endpoint_url=os.environ['MLFLOW_S3_ENDPOINT_URL'],
        aws_access_key_id=os.environ['AWS_ACCESS_KEY_ID'],
        aws_secret_access_key=os.environ['AWS_SECRET_ACCESS_KEY']
    )
    
    try:
        # Vérifier si le bucket existe
        s3.head_bucket(Bucket=bucket_name)
        print(f"✅ Bucket '{bucket_name}' trouvé.")
    except ClientError as e:
        error_code = int(e.response['Error']['Code'])
        if error_code == 404:
            print(f"⚠️ Bucket '{bucket_name}' introuvable. Création en cours...")
            try:
                s3.create_bucket(Bucket=bucket_name)
                print(f"✅ Bucket '{bucket_name}' créé avec succès !")
            except Exception as create_err:
                print(f"❌ Impossible de créer le bucket : {create_err}")
                raise
        else:
            print(f"❌ Erreur d'accès au bucket (Code {error_code}) : {e}")
            print("👉 Vérifiez que MinIO tourne bien sur le port 9000.")
            raise

# On lance la vérification avant tout le reste
try:
    check_and_create_bucket("mlflow")
except Exception as e:
    print("🛑 ARRÊT CRITIQUE : Impossible de contacter MinIO.")
    # On stop l'exécution ici si MinIO n'est pas prêt
    raise e

🔍 Vérification de l'accès MinIO sur http://localhost:9000...
⚠️ Bucket 'mlflow' introuvable. Création en cours...
✅ Bucket 'mlflow' créé avec succès !


In [29]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import pandas as pd
from PIL import Image
import numpy as np
import mlflow
import mlflow.pytorch
from mlflow.models import infer_signature
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from pydantic import BaseModel, Field, field_validator
from typing import List, Dict, Optional

In [30]:

# --- Configuration Globale ---
CSV_PATH = "../annotations.csv"
IMG_DIR = "../dataset_prepro/"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [31]:

# =============================================================================
# 1. DÉFINITION DE LA STRUCTURE DE L'ARBRE (PYDANTIC)
# =============================================================================

class TreeNodeConfig(BaseModel):
    """
    Définit un noeud dans l'arbre du réseau de neurones.
    Récursif : un noeud peut avoir des 'branches' qui sont elles-mêmes des noeuds.
    """
    layers: List[int] = Field(default_factory=list, description="Liste des tailles des couches Dense pour ce segment")
    dropout: float = Field(default=0.0, ge=0.0, le=1.0, description="Taux de dropout pour ce segment")
    branches: Dict[str, 'TreeNodeConfig'] = Field(default_factory=dict, description="Sous-branches (Nom -> Config)")
    tasks: Dict[str, int] = Field(default_factory=dict, description="Têtes de sortie finales (Nom Tâche -> Nb Classes)")

    @field_validator('layers')
    def check_layers(cls, v):
        if any(x <= 0 for x in v): raise ValueError("Les couches doivent avoir une taille > 0")
        return v

# Permet la récursivité de la définition Pydantic
TreeNodeConfig.model_rebuild()

In [32]:

# =============================================================================
# 2. DATASET (ROBUSTE)
# =============================================================================

class DatasetFaces(Dataset):
    def __init__(self, csv_file, img_dir, transform=None, file_extension='.jpg'):
        self.annotations = pd.read_csv(csv_file, index_col='filename')
        self.img_names = self.annotations.index.tolist()
        self.img_dir = img_dir
        self.transform = transform
        self.file_extension = file_extension

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_name_base = self.img_names[idx]
        # Gestion simple des extensions multiples si besoin
        img_path = os.path.join(self.img_dir, img_name_base + self.file_extension)
        
        try:
            image = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
            # Fallback : Image noire pour éviter le crash pendant l'opti, mais à monitorer
            image = Image.new('RGB', (64, 64)) 

        # Récupération des labels (assurons-nous que l'ordre correspond au CSV)
        labels = self.annotations.loc[img_name_base].values.astype(float)
        labels = torch.tensor(labels, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, labels

def get_data_loaders(batch_size):
    data_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    # Attention à l'extension fichier
    full_dataset = DatasetFaces(CSV_PATH, IMG_DIR, transform=data_transform, file_extension=".png")
    
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size
    
    gen = torch.Generator().manual_seed(42)
    train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=gen)
    
    return DataLoader(train_dataset, batch_size=batch_size, shuffle=True), DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [33]:

# =============================================================================
# 3. MODÈLE DYNAMIQUE (CNN + ARBRE RECURSIF)
# =============================================================================

class DynamicTreeBranch(nn.Module):
    def __init__(self, in_features: int, config: TreeNodeConfig):
        super().__init__()
        self.layers_seq = nn.Sequential()
        current_size = in_features
        
        # Construction des couches linéaires locales
        for i, hidden_size in enumerate(config.layers):
            self.layers_seq.add_module(f"fc_{i}", nn.Linear(current_size, hidden_size))
            self.layers_seq.add_module(f"act_{i}", nn.ReLU())
            if config.dropout > 0:
                self.layers_seq.add_module(f"drop_{i}", nn.Dropout(config.dropout))
            current_size = hidden_size
            
        self.branches = nn.ModuleDict()
        self.tasks = nn.ModuleDict()
        
        # Instanciation récursive des branches
        for name, branch_cfg in config.branches.items():
            self.branches[name] = DynamicTreeBranch(current_size, branch_cfg)
            
        # Instanciation des têtes de lecture (Tasks)
        for name, num_classes in config.tasks.items():
            self.tasks[name] = nn.Linear(current_size, num_classes)

    def forward(self, x):
        # 1. Passage dans les couches locales
        x = self.layers_seq(x)
        
        results = {}
        
        # 2. Collecte récursive des résultats des sous-branches
        for branch in self.branches.values():
            results.update(branch(x)) # Merge des dictionnaires
            
        # 3. Calcul des tâches locales
        for name, task_layer in self.tasks.items():
            results[name] = task_layer(x)
            
        return results

In [34]:

class CNN(nn.Module):
    def __init__(self, filters_list: List[int], tree_structure: TreeNodeConfig):
        super().__init__()
        # Sauvegarde de la config pour MLflow (facilite le reload)
        self.filters_list = filters_list
        self.tree_structure_dict = tree_structure.model_dump() 
        
        # Partie Convolutive
        layers = []
        in_channels = 3
        for i, out_channels in enumerate(filters_list):
            layers.append(nn.Conv2d(in_channels, out_channels, 3, padding=1))
            layers.append(nn.BatchNorm2d(out_channels)) # Ajout BatchNorm pour stabilité
            layers.append(nn.ReLU())
            if i > 0: 
                layers.append(nn.MaxPool2d(2))
            in_channels = out_channels
        self.conv = nn.Sequential(*layers)
        
        # Calcul taille Flatten
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 64, 64)
            out = self.conv(dummy)
            self.flattened_size = out.view(1, -1).size(1)
            
        # Partie Arbre
        self.tree = DynamicTreeBranch(self.flattened_size, tree_structure)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.tree(x)

In [35]:

# =============================================================================
# 4. FONCTION OBJECTIVE HYPEROPT (TESTS MASSIFS)
# =============================================================================

def objective(params):
    # Initialisation Run MLflow
    with mlflow.start_run(nested=True):
        
        # --- A. Décodage des Paramètres Hyperopt ---
        lr = params['lr']
        batch_size = int(params['batch_size'])
        epochs = int(params['epochs']) # Peut être réduit pour les tests rapides
        dropout = params['dropout']
        
        # Architecture CNN
        n_conv_layers = int(params['n_conv_layers'])
        base_filters = int(params['base_filters'])
        filters_list = [base_filters * (2**i) for i in range(n_conv_layers)]
        
        # --- B. Génération de la Stratégie d'Arbre ---
        # C'est ici que l'on teste différentes structures d'arbre
        strategy = params['tree_strategy']
        fc_base = int(params['fc_base_size'])
        
        # Définition des têtes (Fixe, requis par l'utilisateur)
        tasks_visage = {"barbe": 1, "moustache": 1, "lunettes": 1}
        task_taille = {"taille_cheveux": 3}
        task_couleur = {"couleur_cheveux": 5}

        if strategy == "hierarchical_user":
            # La structure demandée spécifiquement : Visage | (Cheveux -> Forme | Couleur)
            tree_config = TreeNodeConfig(
                layers=[fc_base],
                dropout=dropout,
                branches={
                    "visage": TreeNodeConfig(
                        layers=[int(fc_base/2)],
                        tasks=tasks_visage
                    ),
                    "cheveux": TreeNodeConfig(
                        layers=[int(fc_base/2)],
                        branches={ # Sous-branches cheveux
                            "forme": TreeNodeConfig(layers=[int(fc_base/4)], tasks=task_taille),
                            "couleur": TreeNodeConfig(layers=[int(fc_base/4)], tasks=task_couleur)
                        }
                    )
                }
            )
            
        elif strategy == "flat_split":
            # Tout part du tronc commun : Visage | Taille | Couleur (indépendants)
            tree_config = TreeNodeConfig(
                layers=[fc_base],
                dropout=dropout,
                branches={
                    "visage": TreeNodeConfig(layers=[int(fc_base/2)], tasks=tasks_visage),
                    "taille": TreeNodeConfig(layers=[int(fc_base/2)], tasks=task_taille),
                    "couleur": TreeNodeConfig(layers=[int(fc_base/2)], tasks=task_couleur)
                }
            )
            
        elif strategy == "deep_shared":
            # Tronc commun très profond, séparation tardive
            tree_config = TreeNodeConfig(
                layers=[fc_base, int(fc_base/2)], # Deux couches communes
                dropout=dropout,
                branches={
                    "all_heads": TreeNodeConfig(
                        layers=[int(fc_base/4)],
                        tasks={**tasks_visage, **task_taille, **task_couleur} # Toutes les tâches au même niveau
                    )
                }
            )

        # Log des params complexes en JSON strings pour lisibilité
        mlflow.log_params({k: v for k, v in params.items() if k != 'tree_structure'})
        mlflow.log_dict(tree_config.model_dump(), "tree_config.json")
        
        print(f"\n🏗️  Testing Strategy: {strategy} | FC: {fc_base} | LR: {lr:.1e}")

        # --- C. Préparation Données & Modèle ---
        try:
            train_loader, test_loader = get_data_loaders(batch_size)
        except Exception as e:
            print(f"Data Error: {e}")
            return {'loss': float('inf'), 'status': STATUS_OK}

        model = CNN(filters_list, tree_config).to(DEVICE)
        optimizer = optim.Adam(model.parameters(), lr=lr)
        
        criterion_bin = nn.BCEWithLogitsLoss()
        criterion_multi = nn.CrossEntropyLoss()

        # Mapping Label CSV -> Tâche
        # (idx_start, idx_end_exclusive, type)
        task_mapping = {
            "barbe":           (0, 1, 'bin'),
            "moustache":       (1, 2, 'bin'),
            "lunettes":        (2, 3, 'bin'),
            "taille_cheveux":  (3, 6, 'multi'),
            "couleur_cheveux": (6, 11, 'multi')
        }

        # --- D. Boucle d'Entraînement ---
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            model.train()
            running_loss = 0.0
            
            for images, labels in train_loader:
                images, labels = images.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                
                outputs = model(images) # Dict de tensors
                loss = 0
                
                for name, (start, end, mode) in task_mapping.items():
                    pred = outputs[name]
                    if mode == 'bin':
                        target = labels[:, start] # shape [N]
                        loss += criterion_bin(pred.squeeze(), target)
                    else:
                        target_chunk = labels[:, start:end]
                        target_indices = torch.argmax(target_chunk, dim=1)
                        loss += criterion_multi(pred, target_indices)
                
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            
            # --- E. Validation ---
            model.eval()
            val_loss = 0.0
            accuracies = {k: 0 for k in task_mapping}
            total = 0
            
            with torch.no_grad():
                for images, labels in test_loader:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)
                    outputs = model(images)
                    total += images.size(0)
                    
                    for name, (start, end, mode) in task_mapping.items():
                        pred = outputs[name]
                        if mode == 'bin':
                            target = labels[:, start]
                            val_loss += criterion_bin(pred.squeeze(), target).item()
                            acc_pred = (torch.sigmoid(pred.squeeze()) > 0.5).float()
                            accuracies[name] += (acc_pred == target).sum().item()
                        else:
                            target_indices = torch.argmax(labels[:, start:end], dim=1)
                            val_loss += criterion_multi(pred, target_indices).item()
                            acc_pred = torch.argmax(pred, dim=1)
                            accuracies[name] += (acc_pred == target_indices).sum().item()
            
            avg_val_loss = val_loss / len(test_loader)
            
            # Logging Metriques
            mlflow.log_metric("val_loss", avg_val_loss, step=epoch)
            for name, count in accuracies.items():
                mlflow.log_metric(f"acc_{name}", count/total, step=epoch)
            
            print(f"  Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}")

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss

        # --- F. Sauvegarde Propre MLflow ---
        # On crée une signature Input/Output
        model.eval()
        with torch.no_grad():
            dummy_in = torch.randn(1, 3, 64, 64).to(DEVICE)
            dummy_out_tensors = model(dummy_in)
            # Convert tensors to numpy dict for signature
            dummy_out_np = {k: v.cpu().numpy() for k, v in dummy_out_tensors.items()}
            signature = infer_signature(dummy_in.cpu().numpy(), dummy_out_np)

        # ASTUCE: On loggue le modèle PyTorch standard
        # MLflow va sérialiser l'objet modèle. Comme nous avons stocké 
        # self.tree_structure_dict et self.filters_list dans __init__, 
        # l'objet contient tout ce qu'il faut pour se reconstruire si la classe est présente.
        mlflow.pytorch.log_model(
            model, 
            "model", 
            signature=signature,
            pip_requirements=["torch", "pydantic", "pandas", "numpy"] # Important pour la rejouabilité
        )
        
        return {'loss': best_val_loss, 'status': STATUS_OK}

In [36]:

# =============================================================================
# 5. MAIN : LANCEMENT DES TESTS HYPEROPT
# =============================================================================

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("MTL_Advanced_Architecture_Search")

# Espace de recherche HYPER LARGE
space = {
    'lr': hp.loguniform('lr', np.log(1e-4), np.log(1e-2)),
    'batch_size': hp.choice('batch_size', [32, 64]),
    'epochs': hp.choice('epochs', [5]), # Fixe pour comparer équitablement
    'dropout': hp.uniform('dropout', 0.0, 0.5),
    
    # Paramètres CNN
    'n_conv_layers': hp.choice('n_conv_layers', [2, 3, 4]),
    'base_filters': hp.choice('base_filters', [16, 32]),
    
    # Paramètres Structure Arbre
    'tree_strategy': hp.choice('tree_strategy', ['hierarchical_user', 'flat_split', 'deep_shared']),
    'fc_base_size': hp.choice('fc_base_size', [128, 256, 512])
}

print("🧠 Lancement de l'optimisation massive...")
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50, # Augmenter pour plus de tests
    trials=trials
)

print("\n🏆 Meilleurs hyperparamètres :")
print(best)

print("\n💡 Pour charger le meilleur modèle plus tard :")
print("loaded_model = mlflow.pytorch.load_model('runs:/<RUN_ID>/model')")

2025/11/20 16:53:53 INFO mlflow.tracking.fluent: Experiment with name 'MTL_Advanced_Architecture_Search' does not exist. Creating a new experiment.


🧠 Lancement de l'optimisation massive...
                                                      
🏗️  Testing Strategy: deep_shared | FC: 256 | LR: 5.2e-04
  Epoch 1 Val Loss: 0.2468                            
  Epoch 2 Val Loss: 0.1208                            
  Epoch 3 Val Loss: 0.0661                            
  Epoch 4 Val Loss: 0.0522                            
  Epoch 5 Val Loss: 0.0500                            
  0%|          | 0/50 [00:58<?, ?trial/s, best loss=?]

2025/11/20 16:54:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run sincere-crab-562 at: http://localhost:5000/#/experiments/1/runs/45f380f58d684d5896c88a49a32b6168

🧪 View experiment at: http://localhost:5000/#/experiments/1

                                                                                 
🏗️  Testing Strategy: hierarchical_user | FC: 128 | LR: 1.3e-04
  Epoch 1 Val Loss: 0.6499                                                       
  Epoch 2 Val Loss: 0.2624                                                       
  Epoch 3 Val Loss: 0.1571                                                       
  Epoch 4 Val Loss: 0.1108                                                       
  Epoch 5 Val Loss: 0.0968                                                       
  2%|▏         | 1/50 [02:06<48:35, 59.49s/trial, best loss: 0.04996562909862717]

2025/11/20 16:55:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

🏃 View run chill-ant-509 at: http://localhost:5000/#/experiments/1/runs/d97dde3ebfb549e3932adb2565eb1dce

🧪 View experiment at: http://localhost:5000/#/experiments/1                        

                                                                                    
🏗️  Testing Strategy: flat_split | FC: 128 | LR: 2.0e-03
  Epoch 1 Val Loss: 0.2512                                                          
  Epoch 2 Val Loss: 0.1367                                                          
  Epoch 3 Val Loss: 0.1078                                                          
  Epoch 4 Val Loss: 0.0744                                                          
  Epoch 5 Val Loss: 0.1130                                                          
 96%|█████████▌| 48/50 [1:01:39<02:38, 79.13s/trial, best loss: 0.03753860369040922]

2025/11/20 17:55:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run zealous-colt-405 at: http://localhost:5000/#/experiments/1/runs/9afc0b96b01e415e83182b8a0c45d758

🧪 View experiment at: http://localhost:5000/#/experiments/1                        

                                                                                    
🏗️  Testing Strategy: deep_shared | FC: 128 | LR: 4.2e-04
  Epoch 1 Val Loss: 0.3625                                                          
  Epoch 2 Val Loss: 0.1285                                                          
  Epoch 3 Val Loss: 0.0855                                                          
  Epoch 4 Val Loss: 0.0545                                                          
  Epoch 5 Val Loss: 0.0584                                                          
 98%|█████████▊| 49/50 [1:03:02<01:18, 78.65s/trial, best loss: 0.03753860369040922]

2025/11/20 17:56:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run dazzling-moose-546 at: http://localhost:5000/#/experiments/1/runs/1109cba9fe394ef0aa61383b7e500588

🧪 View experiment at: http://localhost:5000/#/experiments/1                        

100%|██████████| 50/50 [1:03:03<00:00, 75.68s/trial, best loss: 0.03753860369040922]

🏆 Meilleurs hyperparamètres :
{'base_filters': np.int64(1), 'batch_size': np.int64(1), 'dropout': np.float64(0.004051762077490816), 'epochs': np.int64(0), 'fc_base_size': np.int64(0), 'lr': np.float64(0.0004372333975723074), 'n_conv_layers': np.int64(2), 'tree_strategy': np.int64(1)}

💡 Pour charger le meilleur modèle plus tard :
loaded_model = mlflow.pytorch.load_model('runs:/<RUN_ID>/model')
